# MolDisc regression tutorial

This notebook runs the complete **SMILES-X regression -> GPT generation -> property screening -> reporting** workflow on the bundled small-data example: 100 labelled and 1,000 unlabelled molecules in `data/data_1`. Start Jupyter Lab from the repository root and run the notebook unchanged once. Then keep the bundled example intact, copy its data folder, and edit the **User settings** block for your own dataset. The deliberately small model and generation settings are an installation check, not a scientific benchmark; see `data/README.md` for dataset provenance and terms.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd().resolve()
while repo_root.parent != repo_root and not (repo_root / 'moldisc.py').exists():
    repo_root = repo_root.parent
if not (repo_root / 'moldisc.py').exists():
    raise FileNotFoundError('Could not locate the MolDisc repository root.')
sys.path.insert(0, str(repo_root))

from moldisc import moldisc
from rdkit.Chem.Draw import IPythonConsole
# Keep RDKit grid images as saveable PIL objects during the pipeline.
IPythonConsole.UninstallIPythonRenderer()

# USER SETTINGS: keep data/data_1 unchanged; copy it to a new folder, replace both CSVs,
# then change these values. Use -1 to disable a numeric stopping rule.
PROJECT_NAME = 'data_1'             # folder under data/
PROJECT_FOLDER = 'results/tutorial/regression'
DATA_LABEL = 'Aqueous solubility'
DATA_UNITS = 'log10 mol/L'
STOP_AFTER_CYCLES = 1               # cycle endpoint for this tutorial
STOP_AFTER_GENERATED = -1           # valid molecules absent from active inputs and prior cycles
TARGET_PROPERTY_VALUE = None        # e.g. 0.0; direction is 'max' below

print('MolDisc repository detected.')
print(f'Python: {sys.version.split()[0]}')

In [ ]:
labeled = pd.read_csv(repo_root / 'data' / PROJECT_NAME / 'labeled.csv')
unlabeled = pd.read_csv(repo_root / 'data' / PROJECT_NAME / 'unlabeled.csv')
if list(labeled.columns) != ['smiles', 'property']:
    raise ValueError("labeled.csv must have exactly: smiles, property")
if list(unlabeled.columns) != ['smiles']:
    raise ValueError("unlabeled.csv must have exactly: smiles")
display(labeled.head())
print(f'{len(labeled)} labeled and {len(unlabeled)} unlabeled molecules')
display(labeled['property'].describe().to_frame().T)

## Run the smoke-scale workflow

This example stops after one completed cycle. Compact networks, two folds, and canonical-only inference make it a functional test. MolDisc writes generated artifacts under `PROJECT_FOLDER` and does not modify the source CSV files. The first run downloads `distilgpt2`; later runs reuse the repository cache. Saved SMILES-X models are reused only when the signed input, cleaned-table, hyperparameter, source-code, and artifact digests all match; unsigned or mismatched predictors are retrained. For a scientific campaign, choose one primary endpoint before running: `STOP_AFTER_CYCLES`, `STOP_AFTER_GENERATED`, or `TARGET_PROPERTY_VALUE`. The allowed-element, synthetic-accessibility, and collapse checks remain independent safety guards.

In [ ]:
discovered = moldisc(
    project_name=PROJECT_NAME, project_folder=PROJECT_FOLDER,
    data_name='regression_demo', data_label=DATA_LABEL,
    data_units=DATA_UNITS, model_type='regression', scale_output=True,
    augmentation=False, reuse_smilesx_models=True,
    bs_ref=16, lr_ref=3.0, embed_ref=8,
    lstm_ref=8, tdense_ref=8, dense_depth=1,
    k_fold_number=2, n_runs=1, n_epochs=2, n_gpus=0,
    log_verbose=False, train_verbose=0,
    gpt_pretrained_model='distilgpt2', gpt_augmentation=0,
    gpt_initial_epochs=1, gpt_cycle_epochs=1,
    gpt_tr_batch_size=16, gpt_eval_batch_size=16, gpt_patience=2,
    gpt_num_generation=8, gpt_num_attempts=20,
    gpt_generation_batch_size=32,
    gpt_min_carbon_atoms=2, gpt_min_heavy_atoms=3,
    gpt_allowed_elements=['B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'Br', 'I'],
    smilesx_inference_augmentation=False,
    max_generation=-1, max_generated_molecules=STOP_AFTER_GENERATED,
    cycles=STOP_AFTER_CYCLES, target_property_value=TARGET_PROPERTY_VALUE,
    target_property_mode='max', cutoff=0.5, sa_score=10,
    option_show=False, num_mol_top=4, num_mol_bottom=4, random_seed=42,
)
display(discovered)

In [ ]:
run_dir = repo_root / PROJECT_FOLDER / PROJECT_NAME
display_run_dir = run_dir.relative_to(repo_root) if run_dir.is_relative_to(repo_root) else run_dir
print(f'Results directory: {display_run_dir}')
saved = pd.read_csv(run_dir / 'final' / 'generated_smiles.csv')
display(saved)
if (run_dir / 'cycle_metrics.csv').exists():
    metrics = pd.read_csv(run_dir / 'cycle_metrics.csv')
    summary_columns = ['cycle', 'requested', 'generated', 'cumulative_generated', 'selected', 'property_mean', 'property_max', 'generation_validity_rate', 'generation_novelty_rate', 'generation_yield_rate']
    display(metrics[[column for column in summary_columns if column in metrics.columns]])
figure_paths = sorted((run_dir / 'final').glob('top_*_molecules.png'))[:1]
figure_paths += [run_dir / 'final' / 'tanimoto.jpg']
for image_path in figure_paths:
    if image_path.exists():
        display(Image(filename=str(image_path)))

## Editable JSON configuration

`configs/example_regression.json` is a larger but still bounded starting template. Copy it to a new filename before changing it for scientific work. Set unused endpoints to `-1` (or `null` for `target_property_value`), keep at least one deterministic endpoint active, and prespecify the chemical/collapse guards. Inspecting the file below does not launch a campaign.

In [ ]:
example_config = json.loads((repo_root / 'configs' / 'example_regression.json').read_text(encoding='utf-8'))
keys = ['k_fold_number', 'n_runs', 'n_epochs', 'reuse_smilesx_models',
        'gpt_pretrained_model',
        'gpt_num_generation', 'gpt_min_carbon_atoms', 'gpt_min_heavy_atoms',
        'gpt_allowed_elements',
        'cycles', 'max_generated_molecules', 'target_property_value',
        'sa_score', 'collapse_patience', 'random_seed']
display(pd.Series({key: example_config[key] for key in keys}, name='example regression'))

For quality control and resume, launch from a terminal:

```powershell
python scripts/run_campaign.py --config configs/example_regression.json --pause-after-cycles 1
python scripts/analyze_campaign.py --config configs/example_regression.json
python scripts/run_campaign.py --config configs/example_regression.json --resume
```

The `property` column contains ensemble predictions, not experimental measurements. Treat generated structures as hypotheses requiring applicability-domain analysis, chemical and synthesis review, and experimental or higher-fidelity computational validation.